In [1]:
from transcription_pipeline.RateExtraction import AverageAndFit
import pandas as pd
import matplotlib as mpl
mpl.use('TkAgg')

In [4]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
key = '001'

embryo_list = {
    '001': [
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',

    ],

    '002': [
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
    ],

    '003': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        # 'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
        # 'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
    ]
}
test_dataset_name = dataset_folder + embryo_list[key][2]
print('Dataset Path: ' + test_dataset_name)

Dataset Path: /mnt/Data1/Nick/transcription_pipeline/test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01


In [5]:
# Specify here at what frame NC14 starts
nc14_start_frame = 0

# Width of averaging window for time bins
time_bin_width = 2

# Number of bins you want to split the full embryo into
num_bins = 40

# Load in compiled dataframe
dataframe_path = test_dataset_name + '/compiled_dataframe.pkl'
compiled_dataframe = pd.read_pickle(dataframe_path)

In [6]:
compiled_dataframe.head()

,particle,frame,t_s,intensity_from_neighborhood,intensity_std_error_from_neighborhood,x,y,ap,ap90
0,1,"[34, 39, 42, 45, 46, 47, 48, 49, 50, 51, 52, 5...","[72.5955910000205, 82.53665299999714, 88.77964...","[104.12035097493036, 81.97830578512395, 83.082...","[53.37929225700253, 53.47017953838384, 57.9533...","[600.2795472555922, 599.6633007841755, 599.883...","[47.498994746506185, 46.03165211132432, 46.179...","[0.23726241602168038, 0.23755445859599475, 0.2...","[-0.09492943003707735, -0.09644941369939097, -..."
1,2,"[78, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 9...","[161.95579000002147, 167.85077200001479, 169.8...","[220.3791491712707, 201.4196647230321, 206.717...","[57.440801966877935, 55.67358226582613, 53.279...","[683.0001667378026, 683.8773108654434, 683.591...","[96.97089914552355, 96.27502272173149, 96.1246...","[0.20180517788735036, 0.20146018476813737, 0.2...","[-0.04772702394524998, -0.048524214821121965, ..."
2,4,"[29, 30, 32, 33, 35, 37, 38, 39, 40, 41, 42, 4...","[62.65455400002003, 64.67755199998618, 68.8976...","[66.99725698324022, 216.86624000000003, 175.33...","[50.10122217848835, 49.94716913065644, 52.5114...","[740.636297374587, 739.5845824466095, 739.7486...","[168.2423494126858, 167.90366576504545, 167.84...","[0.1761658125543142, 0.1766092478619215, 0.176...","[0.024292503103676713, 0.024001048924255208, 0..."
3,6,"[336, 338, 339, 340, 341, 343, 344, 345, 346, ...","[683.3683059999943, 687.4143659999967, 689.437...","[260.9195580110497, 95.77432748538011, 59.9775...","[44.37378634825406, 45.18937470915508, 48.3634...","[904.0387762009617, 903.2637364122018, 903.712...","[186.71904429070395, 187.8987003489536, 187.86...","[0.10813713875598177, 0.10842763430926829, 0.1...","[0.03327012734847857, 0.03457505240563834, 0.0..."
4,7,"[368, 369, 371, 373, 375, 376, 378, 379, 381, ...","[748.1043769999743, 750.1274070000053, 754.173...","[5.4421625344352655, 209.45433333333332, 204.7...","[57.45250778187088, 54.853487130336795, 52.495...","[573.0044486671862, 572.2163620386324, 572.468...","[104.35699597389187, 103.8005643699527, 102.23...","[0.2470959577688664, 0.24743592269798503, 0.24...","[-0.032691795555266474, -0.03323199240580166, ..."


In [7]:
aafdata = AverageAndFit(compiled_dataframe, nc14_start_frame, time_bin_width, num_bins, test_dataset_name, shift_traces=False)

No previous bin fit checking results detected. Do bin fitting for the dataframe.
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate


/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1416: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.bin_average_fit_dataframe['time_bin_centers'][bin] = time_bin_centers
/mnt/Data1/Nick/transcription_pipe

length of MS2: 49, length of t_interp: 49
Failed to fit average trace 4: Need at least 4 data points to fit half-cycle model


/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1427: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.bin_average_fit_dataframe['bin_fit_result'][bin] = bin_fit_result
/mnt/Data1/Nick/transcription_pipeline

length of MS2: 128, length of t_interp: 128


...

/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1427: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.bin_average_fit_dataframe['bin_fit_result'][bin] = bin_fit_result
/mnt/Data1/Nick/transcription_pipeline

length of MS2: 42, length of t_interp: 42


/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1427: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.bin_average_fit_dataframe['bin_fit_result'][bin] = bin_fit_result
/mnt/Data1/Nick/transcription_pipeline

length of MS2: 55, length of t_interp: 55
Failed to fit average trace 19: Need at least 4 data points to fit half-cycle model
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least one array to concatenate
Error in bin_average_fit_dataframe: need at least

/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1427: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.bin_average_fit_dataframe['bin_fit_result'][bin] = bin_fit_result
/mnt/Data1/Nick/transcription_pipeline

In [ ]:
aafdata.bin_average_fit_dataframe

In [13]:
mpl.use('TkAgg')
aafdata.check_bin_fits()

/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1493: UserWarning: No bin has been left unchecked. Move to the first bin with data
  warn('No bin has been left unchecked. Move to the first bin with data')
can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


In [9]:
aafdata.save_checked_bin_fits()

Checked bin fits saved


In [11]:
aaf_ap, aaf_slope, aaf_slope_err = aafdata.plot_bin_fits()

In [ ]:
aaf_slope

In [ ]:
# Load checked particle fits
fits_path = test_dataset_name + '/bin_fits_checked.pkl'
bin_fits_checked = pd.read_pickle(fits_path)

In [ ]:
bin_fits_checked

In [ ]:
print(bin_fits_checked['bin_fit_result_modified'][6])

In [ ]:
print(bin_fits_checked['bin_fit_slope'])